# Best-of-N

**Paper**: [WebGPT: Browser-assisted question-answering with human feedback](https://arxiv.org/abs/2112.09332)

**Authors**: Reiichiro Nakano, Jacob Hilton, Suchir Balaji, Jeff Wu, Long Ouyang, Christina Kim, Christopher Hesse, Shantanu Jain, Vineet Kosaraju, William Saunders, Xu Jiang, Karl Cobbe, Tyna Eloundou, Gretchen Krueger, Kevin Button, Matthew Knight, Benjamin Chess, John Schulman

Best-of-N sampling is the standard inference-time alignment baseline. It samples several full-length continuations from the base model and returns the single highest-scoring one under a supplied sequence scorer. Pairing the scorer with a majority-vote scorer recovers self-consistency; pairing it with a metric scorer gives metric-guided reranking.

Best-of-N is a decoding driver built on the generic search driver, mapping onto a single search iteration (`num_candidates=n`, `keep_k=1`, `max_iterations=1`, `propose_mode="sample"`) whose one segment spans the whole `max_new_tokens` budget. Each sampled continuation is a full rollout, so any composed logits processor (for example RAD) steers every sample. Parameters for the scorer travel to it at inference time via `runtime_kwargs={"reward_params": {...}}`.

## Method parameters

| parameter | type | description |
| --------- | ---- | ----------- |
| `n` | `int` | Number of full-length continuations to sample and rank |
| `scorer` | `Callable` | A sequence scorer `(prompt, continuations, params) -> list[float]`; the highest-scoring sample is returned |

## Setup

If running this from a Google Colab notebook, uncomment the clone cell below. It is not necessary when running from a virtual environment where the package is already installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub:

In [ ]:
# !pip install -q python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: reranking by keyword coverage

The scorer is any callable `(prompt, continuations, params) -> list[float]`, where `params` is whatever was passed as `reward_params` at generation time. We define a scorer that counts how many required keywords a continuation covers, then ask for a single sentence that works in all of them.

In [ ]:

from transformers import AutoTokenizer, set_seed

from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.output_control.best_of_n.control import BestOfN

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

The scorer rewards one point per covered keyword. With ten required words and a 56-token budget, a single sample always drops a few of them, which gives reranking something to do.

In [4]:
def keyword_coverage(prompt, continuations, params):
    terms = [t.lower() for t in params.get("key_terms", [])]
    return [float(sum(term in c.lower() for term in terms)) for c in continuations]


KEY_TERMS = ["cat", "couch", "sun", "tea", "nap", "book", "rain", "socks", "lamp", "blanket"]

### Baseline: a single sample (`n=1`)

With one candidate, taking the argmax over one score is a no-op, so `n=1` is plain sampling. We fix the seed so the runs below are comparable, and print the scorer's verdict alongside the output.

In [5]:
pipeline_n1 = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=1, scorer=keyword_coverage)],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline_n1.steer()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
prompt = (
    "Write one sentence about a lazy afternoon at home that mentions all of these words: "
    "cat, couch, sun, tea, nap, book, rain, socks, lamp, blanket."
)
chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt").to(pipeline_n1.model.device)

set_seed(42)
output = pipeline_n1.generate(
    input_ids=inputs["input_ids"],
    runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
    max_new_tokens=56,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
text_n1 = tokenizer.decode(output[0], skip_special_tokens=True)
print(text_n1)
print("\nkeyword score:", keyword_coverage(prompt, [text_n1], {"key_terms": KEY_TERMS})[0])

On a rainy afternoon, my fluffy cat curled up on the comfy couch to enjoy a warm cup of tea while I napped under a cozy blanket, surrounded by books and illuminated by the soft glow of the lamp.

keyword score: 8.0


A single sample is at the mercy of the sampling path it happens to take; the score records how many of the ten keywords it covered.

### Best of 8

Same seed, same prompt, but the driver now proposes eight full continuations, scores each with `keyword_coverage`, and returns the argmax.

In [6]:
pipeline_n8 = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=8, scorer=keyword_coverage)],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline_n8.steer()

set_seed(42)
output = pipeline_n8.generate(
    input_ids=inputs["input_ids"].to(pipeline_n8.model.device),
    runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
    max_new_tokens=56,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
text_n8 = tokenizer.decode(output[0], skip_special_tokens=True)
print(text_n8)
print("\nkeyword score:", keyword_coverage(prompt, [text_n8], {"key_terms": KEY_TERMS})[0])

On a rainy afternoon, I lazily snuggled on the cozy couch with my favorite cat, sipping hot tea while napping under an oversized blanket next to a crackling lamp, surrounded by piles of books and enjoying the gentle sound of rain outside.

keyword score: 8.0


With eight candidates to choose from, the returned continuation covers more of the required keywords than the single sample did. Nothing about the model changed; the improvement comes entirely from selection.

### What the driver does internally

One search iteration is nothing more than propose, score, keep. The cell below reproduces it directly against the same model: sample eight continuations with `num_return_sequences=8`, score them with the same scorer, and take the argmax. `BestOfN` automates exactly this loop, and generalizes it, since the pipeline's composed logits processors and stopping criteria apply to every rollout.

In [7]:
set_seed(42)
rollouts = pipeline_n8.model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=56,
    do_sample=True,
    num_return_sequences=8,
    pad_token_id=tokenizer.eos_token_id,
)
continuations = tokenizer.batch_decode(rollouts[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
scores = keyword_coverage(prompt, continuations, {"key_terms": KEY_TERMS})

for score, continuation in sorted(zip(scores, continuations), reverse=True):
    print(f"[{score:.0f}] {continuation}")

[8] On a rainy afternoon, I lazily snuggled on the cozy couch with my favorite cat, sipping hot tea while napping under an oversized blanket next to a crackling lamp, surrounded by piles of books and enjoying the gentle sound of rain outside.
[8] On a lazy afternoon at home with my fluffy cat curled up on the cozy couch, I sipped steaming tea while napping under a blanket and reading a book by the warm lamp, enjoying the gentle rain outside through the open window as I snuggled deeper into my fluffy
[7] On a lazy afternoon, I snuggled with my favorite cat on the soft couch, sipped some steaming tea while reading a cozy book under the warm glow of the lamp, and enjoyed the gentle sound of rain outside as I napped peacefully in front of the cozy fireplace
[7] On a lazy afternoon at home, I curled up on the cozy couch with my favorite book, sipped some warm tea while reading, napped under a soft blanket, and watched the rain outside through the window, all thanks to my fluffy cat who snug

The spread across the eight samples is the whole story of best-of-N. Every rollout misses at least a couple of the ten words, the best cover the most, and the driver simply keeps the top row of this list.

### Scaling `n`

Each candidate is a full rollout, so best-of-N costs `n` times the decode compute of a single generation. The sweep below reads out what that compute buys on this task.

In [8]:
for n in [1, 4, 16]:
    sweep_pipeline = SteeringPipeline(
        model_name_or_path=MODEL_NAME,
        controls=[BestOfN(n=n, scorer=keyword_coverage)],
        device_map="auto",
        hf_model_kwargs={"dtype": "auto"},
    )
    sweep_pipeline.steer()
    set_seed(42)
    output = sweep_pipeline.generate(
        input_ids=inputs["input_ids"].to(sweep_pipeline.model.device),
        runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
        max_new_tokens=56,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    score = keyword_coverage(prompt, [text], {"key_terms": KEY_TERMS})[0]
    print(f"n={n:>2}  winner score: {score:.0f}/10")

n= 1  winner score: 8/10


n= 4  winner score: 8/10


n=16  winner score: 9/10


The winner's score improves with `n` and then saturates; past a point, a larger pool mostly resamples the same near-best coverage instead of finding sentences that work in every word. This score-versus-compute curve is the practical dial of the method.

## Example: self-consistency with `MajorityVoteScorer`

Swapping the scorer changes the method. `MajorityVoteScorer` scores each continuation by how many of the others share its extracted answer, so best-of-N with this scorer returns a continuation carrying the plurality answer over `n` sampled reasoning paths. This is self-consistency (Wang et al., 2022), obtained purely as a scorer choice. The scorer takes an `answer_extractor`; here we anchor on the response's final `Answer:` line, with a last-number fallback.

In [9]:
import re

from aisteer360.algorithms.output_control.common.scorers import MajorityVoteScorer


def extract_answer(text: str) -> str:
    match = re.search(r"Answer:\s*\$?(-?\d+(?:\.\d+)?)", text)
    if match:
        return str(float(match.group(1)))
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return str(float(numbers[-1])) if numbers else ""


majority_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=8, scorer=MajorityVoteScorer(answer_extractor=extract_answer))],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
majority_pipeline.steer()

math_prompt = (
    "If it takes 5 machines 5 minutes to make 5 widgets, how many minutes would it take 100 machines "
    'to make 100 widgets? Work through it step by step, then end your response with "Answer: <number>".'
)
math_chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": math_prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
math_inputs = tokenizer(math_chat, return_tensors="pt").to(majority_pipeline.model.device)

set_seed(42)
output = majority_pipeline.generate(
    input_ids=math_inputs["input_ids"],
    max_new_tokens=300,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,
)
majority_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(majority_text)
print("\nextracted answer:", extract_answer(majority_text))

Firstly, let's analyze the given information:

- It takes 5 machines 5 minutes to make 5 widgets.

From this, we can deduce that:
- All 5 machines working together make 5 widgets in 5 minutes.
- Therefore, each machine makes 1 widget in 5 minutes when all 5 machines are working together.

Now, if there are 100 machines instead of 5 and they need to make 100 widgets, we follow these steps:

1. Since one machine makes 1 widget in 5 minutes, 100 machines will also make 1 widget in 5 minutes (because they are working simultaneously).

2. To find out how long it takes for 100 machines to make 100 widgets, we note that since one machine can make 1 widget in 5 minutes, 100 machines can make 100 widgets in the same amount of time because they are all contributing equally.

Therefore, it will still take **5 minutes** for 100 machines to make 100 widgets.

Answer: 5

extracted answer: 5.0


### Comparison: a single greedy answer

The self-consistency claim is that the plurality over sampled reasoning paths beats the single path greedy decoding commits to. For the comparison we decode the same prompt greedily, without the driver.

In [10]:
greedy_ids = majority_pipeline.model.generate(
    input_ids=math_inputs["input_ids"],
    attention_mask=math_inputs["attention_mask"],
    max_new_tokens=300,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
greedy_text = tokenizer.decode(greedy_ids[0][math_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(greedy_text)
print("\nextracted answer:", extract_answer(greedy_text))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


To solve this problem, let's break it down step by step:

1. **Understand the given information:**
   - 5 machines can make 5 widgets in 5 minutes.

2. **Determine the rate of production for one machine:**
   - Since 5 machines can produce 5 widgets in 5 minutes, each machine produces \( \frac{5 \text{ widgets}}{5 \text{ machines} \times 5 \text{ minutes}} = 1 \text{ widget per minute per machine} \).

3. **Calculate the time required for 100 machines to make 100 widgets:**
   - If one machine can produce 1 widget in 1 minute, then 100 machines will produce 100 widgets in 1 minute.

Therefore, if 100 machines work together at the same rate as one machine, they will also be able to produce 100 widgets in 1 minute.

**Answer: 1**

extracted answer: 1.0


The correct answer is 5 minutes (each machine makes one widget in 5 minutes, so 100 machines make 100 widgets in the same 5 minutes). Both routes land on it here: the greedy path solves this instance, and the majority scorer returns a continuation from the plurality cluster of sampled paths. The value of self-consistency is robustness. Individual samples do occasionally fall for the trap readings, and as problems harden past what the single greedy path reliably solves, the plurality over sampled paths keeps winning (Wang et al.'s result).

### Takeaway

Best-of-N is the first thing to try when you can score what you want: it needs no training, composes with everything, and costs a transparent `n` full decodes per output. The scorer is the method, as this notebook shows twice with the same driver (keyword reranking, then self-consistency via `MajorityVoteScorer`; the shipped scorers live in `aisteer360.algorithms.output_control.common.scorers`).

Because every candidate is a full rollout through the composed stacks, a step-level control steers all `n` samples; running RAD under `BestOfN` reranks already-detoxified candidates ([rad.ipynb](rad.ipynb)). For iterative segment-level search with the same scorer contract, see DeAL ([deal.ipynb](deal.ipynb)). See the [output control](https://ibm.github.io/AISteer360/concepts/controls/#output-control) section of the docs for the full family.